In [ ]:
# 13 - Ensembles y Gradient Boosting

##Este notebook corresponde al Rol 3: Ensemble Engineer del Sprint 4.

##El objetivo es entrenar y optimizar modelos avanzados de Gradient Boosting, específicamente XGBoost y LightGBM, usando el dataset de entrenamiento procesado.

##El test set no se utiliza en este notebook, ya que será reservado para la validación final del modelo seleccionado.

In [1]:
import os
import time
import joblib
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

In [2]:
DATA_PATH = "../data/processed/X_train_balanced_final.csv"

MODELS_DIR = "../models"
RESULTS_PATH = "../models/ensemble_results_xgb_lgbm.csv"

XGB_BASE_PATH = "../models/xgboost_base.pkl"
LGBM_BASE_PATH = "../models/lightgbm_base.pkl"
XGB_TUNED_PATH = "../models/tuned_xgboost.pkl"
LGBM_TUNED_PATH = "../models/tuned_lightgbm.pkl"

os.makedirs(MODELS_DIR, exist_ok=True)

In [3]:
df = pd.read_csv(DATA_PATH)

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras columnas:")
print(df.columns.tolist()[:15])

print("\nDistribución de Revenue:")
print(df["Revenue"].value_counts())

Dimensiones del dataset: (16476, 55)

Primeras columnas:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType']

Distribución de Revenue:
Revenue
1    8238
0    8238
Name: count, dtype: int64


In [4]:
TARGET_COL = "Revenue"

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print("X:", X.shape)
print("y:", y.shape)

print("\nDistribución porcentual del target:")
print(y.value_counts(normalize=True).round(3))

X: (16476, 54)
y: (16476,)

Distribución porcentual del target:
Revenue
1    0.5
0    0.5
Name: proportion, dtype: float64


In [5]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc"
}

In [6]:
def evaluate_model_cv(model, X, y, cv, scoring, model_name, version):
    """
    Evalúa un modelo usando validación cruzada estratificada.
    Devuelve métricas promedio y desviación estándar.
    """
    start_time = time.time()
    
    scores = cross_validate(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False
    )
    
    elapsed_time = time.time() - start_time
    
    result = {
        "model": model_name,
        "version": version,
        "accuracy_cv_mean": scores["test_accuracy"].mean(),
        "accuracy_cv_std": scores["test_accuracy"].std(),
        "f1_cv_mean": scores["test_f1"].mean(),
        "f1_cv_std": scores["test_f1"].std(),
        "precision_cv_mean": scores["test_precision"].mean(),
        "precision_cv_std": scores["test_precision"].std(),
        "recall_cv_mean": scores["test_recall"].mean(),
        "recall_cv_std": scores["test_recall"].std(),
        "roc_auc_cv_mean": scores["test_roc_auc"].mean(),
        "roc_auc_cv_std": scores["test_roc_auc"].std(),
        "time_seconds": round(elapsed_time, 2)
    }
    
    return result

In [7]:
results = []

In [8]:
print("Todo listo para entrenar.")
print("Cantidad de filas:", X.shape[0])
print("Cantidad de variables:", X.shape[1])

Todo listo para entrenar.
Cantidad de filas: 16476
Cantidad de variables: 54


In [ ]:
## Modelo 1: XGBoost base

##Se entrena un modelo XGBoost con hiperparámetros iniciales razonables.  
##Este modelo servirá como punto de comparación frente a la versión tuneada.

In [9]:
xgb_base = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

xgb_base_result = evaluate_model_cv(
    model=xgb_base,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    model_name="XGBoost",
    version="base"
)

results.append(xgb_base_result)

pd.DataFrame(results)

,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds
0,XGBoost,base,0.941673,0.004099,0.941779,0.004134,0.940017,0.003931,0.943553,0.004792,0.987602,0.001373,10.94


In [13]:
if not os.path.exists(XGB_BASE_PATH):
    xgb_base.fit(X, y)
    joblib.dump(xgb_base, XGB_BASE_PATH)
    print(f"Modelo XGBoost base guardado en: {XGB_BASE_PATH}")
else:
    print(f"El modelo XGBoost base ya existe en: {XGB_BASE_PATH}")

Modelo XGBoost base guardado en: ../models/xgboost_base.pkl


In [ ]:
## Modelo 2: LightGBM base

##Se entrena un modelo LightGBM con hiperparámetros iniciales razonables.  
##Este modelo también pertenece a la familia de Gradient Boosting y se comparará contra XGBoost.

In [10]:
lgbm_base = LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

lgbm_base_result = evaluate_model_cv(
    model=lgbm_base,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    model_name="LightGBM",
    version="base"
)

results.append(lgbm_base_result)

pd.DataFrame(results)

,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds
0,XGBoost,base,0.941673,0.004099,0.941779,0.004134,0.940017,0.003931,0.943553,0.004792,0.987602,0.001373,10.94
1,LightGBM,base,0.939791,0.003349,0.939847,0.003388,0.938942,0.003385,0.940761,0.004282,0.986877,0.001859,10.08


In [15]:
if not os.path.exists(LGBM_BASE_PATH):
    lgbm_base.fit(X, y)
    joblib.dump(lgbm_base, LGBM_BASE_PATH)
    print(f"Modelo LightGBM base guardado en: {LGBM_BASE_PATH}")
else:
    print(f"El modelo LightGBM base ya existe en: {LGBM_BASE_PATH}")

Modelo LightGBM base guardado en: ../models/lightgbm_base.pkl


In [ ]:
## Tuning de modelos

##Luego de entrenar los modelos base, se aplica RandomizedSearchCV para optimizar hiperparámetros de XGBoost y LightGBM.

##Se utiliza F1-score como métrica principal, ya que permite equilibrar precision y recall en un problema de clasificación binaria.

In [11]:
xgb_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1
)

param_dist_xgb = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(3, 10),
    "learning_rate": loguniform(0.01, 0.3),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "gamma": uniform(0, 0.5)
}

random_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=True
)

random_xgb.fit(X, y)

print("Mejor F1 CV XGBoost:", random_xgb.best_score_)
print("Mejores parámetros XGBoost:")
print(random_xgb.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejor F1 CV XGBoost: 0.9450283706435961
Mejores parámetros XGBoost:
{'colsample_bytree': np.float64(0.7297380084021096), 'gamma': np.float64(0.061043977350336676), 'learning_rate': np.float64(0.03359658305322321), 'max_depth': 8, 'min_child_weight': 1, 'n_estimators': 484, 'subsample': np.float64(0.6911740650167767)}


In [12]:
best_xgb = random_xgb.best_estimator_

xgb_tuned_result = evaluate_model_cv(
    model=best_xgb,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    model_name="XGBoost",
    version="tuned"
)

xgb_tuned_result["best_params"] = str(random_xgb.best_params_)
xgb_tuned_result["model_path"] = XGB_TUNED_PATH

results.append(xgb_tuned_result)

pd.DataFrame(results)

,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
0,XGBoost,base,0.941673,0.004099,0.941779,0.004134,0.940017,0.003931,0.943553,0.004792,0.987602,0.001373,10.94,NaN,NaN
1,LightGBM,base,0.939791,0.003349,0.939847,0.003388,0.938942,0.003385,0.940761,0.004282,0.986877,0.001859,10.08,NaN,NaN
2,XGBoost,tuned,0.944646,0.003557,0.945028,0.003510,0.938584,0.004103,0.951566,0.003295,0.988488,0.001551,15.51,{'colsample_bytree': np.float64(0.729738008402...,../models/tuned_xgboost.pkl


In [13]:
if not os.path.exists(XGB_TUNED_PATH):
    best_xgb.fit(X, y)
    joblib.dump(best_xgb, XGB_TUNED_PATH)
    print(f"Modelo XGBoost tuneado guardado en: {XGB_TUNED_PATH}")
else:
    print(f"El modelo XGBoost tuneado ya existe en: {XGB_TUNED_PATH}")

Modelo XGBoost tuneado guardado en: ../models/tuned_xgboost.pkl


In [ ]:
## Tuning de LightGBM

##Se aplica RandomizedSearchCV para optimizar hiperparámetros de LightGBM.  
##Se mantiene F1-score como métrica principal y validación cruzada estratificada de 5 folds.

In [14]:
lgbm_model = LGBMClassifier(
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

param_dist_lgbm = {
    "n_estimators": randint(100, 500),
    "max_depth": randint(3, 12),
    "learning_rate": loguniform(0.01, 0.3),
    "num_leaves": randint(15, 80),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_samples": randint(10, 80),
    "reg_alpha": loguniform(1e-4, 1.0),
    "reg_lambda": loguniform(1e-4, 1.0)
}

random_lgbm = RandomizedSearchCV(
    estimator=lgbm_model,
    param_distributions=param_dist_lgbm,
    n_iter=20,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    return_train_score=True
)

random_lgbm.fit(X, y)

print("Mejor F1 CV LightGBM:", random_lgbm.best_score_)
print("Mejores parámetros LightGBM:")
print(random_lgbm.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Mejor F1 CV LightGBM: 0.9434672943432061
Mejores parámetros LightGBM:
{'colsample_bytree': np.float64(0.8281775897621597), 'learning_rate': np.float64(0.05879431781724018), 'max_depth': 10, 'min_child_samples': 62, 'n_estimators': 379, 'num_leaves': 40, 'reg_alpha': np.float64(0.7261414516028828), 'reg_lambda': np.float64(0.026800135558226978), 'subsample': np.float64(0.7103996728090174)}


In [15]:
best_lgbm = random_lgbm.best_estimator_

lgbm_tuned_result = evaluate_model_cv(
    model=best_lgbm,
    X=X,
    y=y,
    cv=cv,
    scoring=scoring,
    model_name="LightGBM",
    version="tuned"
)

lgbm_tuned_result["best_params"] = str(random_lgbm.best_params_)
lgbm_tuned_result["model_path"] = LGBM_TUNED_PATH

results.append(lgbm_tuned_result)

pd.DataFrame(results)

,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
0,XGBoost,base,0.941673,0.004099,0.941779,0.004134,0.940017,0.003931,0.943553,0.004792,0.987602,0.001373,10.94,NaN,NaN
1,LightGBM,base,0.939791,0.003349,0.939847,0.003388,0.938942,0.003385,0.940761,0.004282,0.986877,0.001859,10.08,NaN,NaN
2,XGBoost,tuned,0.944646,0.003557,0.945028,0.003510,0.938584,0.004103,0.951566,0.003295,0.988488,0.001551,15.51,{'colsample_bytree': np.float64(0.729738008402...,../models/tuned_xgboost.pkl
3,LightGBM,tuned,0.943433,0.002958,0.943467,0.002950,0.942898,0.003286,0.944039,0.002894,0.987803,0.001352,10.70,{'colsample_bytree': np.float64(0.828177589762...,../models/tuned_lightgbm.pkl


In [16]:
if not os.path.exists(LGBM_TUNED_PATH):
    best_lgbm.fit(X, y)
    joblib.dump(best_lgbm, LGBM_TUNED_PATH)
    print(f"Modelo LightGBM tuneado guardado en: {LGBM_TUNED_PATH}")
else:
    print(f"El modelo LightGBM tuneado ya existe en: {LGBM_TUNED_PATH}")

Modelo LightGBM tuneado guardado en: ../models/tuned_lightgbm.pkl


In [17]:
results_df = pd.DataFrame(results)

cols_order = [
    "model",
    "version",
    "accuracy_cv_mean",
    "accuracy_cv_std",
    "f1_cv_mean",
    "f1_cv_std",
    "precision_cv_mean",
    "precision_cv_std",
    "recall_cv_mean",
    "recall_cv_std",
    "roc_auc_cv_mean",
    "roc_auc_cv_std",
    "time_seconds",
    "best_params",
    "model_path"
]

for col in cols_order:
    if col not in results_df.columns:
        results_df[col] = ""

results_df = results_df[cols_order]

results_df_sorted = results_df.sort_values(
    by="f1_cv_mean",
    ascending=False
).reset_index(drop=True)

results_df_sorted

,model,version,accuracy_cv_mean,accuracy_cv_std,f1_cv_mean,f1_cv_std,precision_cv_mean,precision_cv_std,recall_cv_mean,recall_cv_std,roc_auc_cv_mean,roc_auc_cv_std,time_seconds,best_params,model_path
0,XGBoost,tuned,0.944646,0.003557,0.945028,0.003510,0.938584,0.004103,0.951566,0.003295,0.988488,0.001551,15.51,{'colsample_bytree': np.float64(0.729738008402...,../models/tuned_xgboost.pkl
1,LightGBM,tuned,0.943433,0.002958,0.943467,0.002950,0.942898,0.003286,0.944039,0.002894,0.987803,0.001352,10.70,{'colsample_bytree': np.float64(0.828177589762...,../models/tuned_lightgbm.pkl
2,XGBoost,base,0.941673,0.004099,0.941779,0.004134,0.940017,0.003931,0.943553,0.004792,0.987602,0.001373,10.94,NaN,NaN
3,LightGBM,base,0.939791,0.003349,0.939847,0.003388,0.938942,0.003385,0.940761,0.004282,0.986877,0.001859,10.08,NaN,NaN


In [18]:
def calculate_improvement(df, model_name, metric="f1_cv_mean"):
    base_value = df[
        (df["model"] == model_name) & 
        (df["version"] == "base")
    ][metric].values[0]
    
    tuned_value = df[
        (df["model"] == model_name) & 
        (df["version"] == "tuned")
    ][metric].values[0]
    
    improvement = ((tuned_value - base_value) / base_value) * 100
    
    return {
        "model": model_name,
        f"{metric}_base": base_value,
        f"{metric}_tuned": tuned_value,
        "improvement_%": round(improvement, 2)
    }

improvement_df = pd.DataFrame([
    calculate_improvement(results_df, "XGBoost"),
    calculate_improvement(results_df, "LightGBM")
])

improvement_df

,model,f1_cv_mean_base,f1_cv_mean_tuned,improvement_%
0,XGBoost,0.941779,0.945028,0.34
1,LightGBM,0.939847,0.943467,0.39


In [19]:
results_df_sorted.to_csv(RESULTS_PATH, index=False)

print(f"Resultados exportados en: {RESULTS_PATH}")

Resultados exportados en: ../models/ensemble_results_xgb_lgbm.csv


In [20]:
best_row = results_df_sorted.iloc[0]

print("Mejor modelo de la parte Ensemble Engineer:")
print("Modelo:", best_row["model"])
print("Versión:", best_row["version"])
print("F1 CV:", round(best_row["f1_cv_mean"], 4))
print("Precision CV:", round(best_row["precision_cv_mean"], 4))
print("Recall CV:", round(best_row["recall_cv_mean"], 4))
print("ROC-AUC CV:", round(best_row["roc_auc_cv_mean"], 4))
print("Ruta del modelo:", best_row["model_path"])

Mejor modelo de la parte Ensemble Engineer:
Modelo: XGBoost
Versión: tuned
F1 CV: 0.945
Precision CV: 0.9386
Recall CV: 0.9516
ROC-AUC CV: 0.9885
Ruta del modelo: ../models/tuned_xgboost.pkl


In [21]:
## Conclusión

##En este notebook se entrenaron y optimizaron dos modelos avanzados de Gradient Boosting: XGBoost y LightGBM.

##Ambos modelos fueron evaluados mediante validación cruzada estratificada de 5 folds, utilizando F1-score como métrica principal. Esta elección permite equilibrar precision y recall en el problema de clasificación binaria de la variable `Revenue`.

##Los resultados muestran que ambos modelos base ya presentaban un rendimiento alto, por lo que el margen de mejora mediante tuning fue reducido. Aun así, el tuning permitió mejorar el F1-score promedio tanto en XGBoost como en LightGBM.

##El mejor desempeño de esta etapa fue obtenido por XGBoost tuneado, ya que alcanzó el mayor F1-score promedio en validación cruzada, junto con el mayor recall y ROC-AUC dentro de los modelos evaluados.

##Los modelos tuneados fueron persistidos en la carpeta `models/`, y los resultados comparativos fueron exportados en `ensemble_results_xgb_lgbm.csv`. Estos outputs podrán ser utilizados por el Experiment Tracker para actualizar el registro general de experimentos y por el Final Validator para seleccionar el modelo final que será evaluado en el test set.

SyntaxError: invalid syntax (3865935732.py, line 3)